# Faster Indicators — Batch Historical Run

Runs the full port call detection and reporting pipeline for 97 pre-defined
4-week windows spanning 2019-01-07 to 2026-06-14. Each window uses its own
ship register snapshot from the central S3 bucket.

**Resumable**: windows with an existing `fi_port_calls.csv` are skipped.
All outputs are saved to `./output/{start_date}_{end_date}/`.

In [1]:
!pip install pandera


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import pickle
import time
import warnings
from functools import partial
from io import StringIO
from pathlib import Path

import cloudpickle
import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast

from shipping_indicators import ports, ships, positions, utils, reporting, livechecks
from shipping_indicators.port_calls.speed_spark import identify_port_call_candidates, dedup_port_calls, column_sets
from shipping_indicators.port_calls.common_spark import check_vessel_in_port_zone, redact_startend_times
from shipping_indicators.port_calls import common_pandas as pc_common_pandas
from shipping_indicators.port_calls import common_spark as pc_common_spark

In [3]:
import sys
import os

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

## Batch Schedule

97 windows of ~4 weeks each. Each entry: (start_date, end_date, ship_register_filename).
Ship registers are sourced from the central bucket:
`s3a://datalab-142496269814-user-bucket/user-sc5k-ochag/ship_register/`

In [4]:
BATCH_SCHEDULE = [
    ('2019-01-07', '2019-02-03', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2019-02-04', '2019-03-03', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2019-03-04', '2019-03-31', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2019-04-01', '2019-04-28', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2019-04-29', '2019-05-26', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2019-05-27', '2019-06-23', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2019-06-24', '2019-07-21', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2019-07-22', '2019-08-18', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2019-08-19', '2019-09-15', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2019-09-16', '2019-10-13', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2019-10-14', '2019-11-10', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2019-11-11', '2019-12-08', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2019-12-09', '2020-01-05', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2020-01-06', '2020-02-02', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2020-02-03', '2020-03-01', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2020-03-02', '2020-03-29', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2020-03-30', '2020-04-26', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2020-04-27', '2020-05-24', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2020-05-25', '2020-06-21', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2020-06-22', '2020-07-19', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2020-07-20', '2020-08-16', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2020-08-17', '2020-09-13', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2020-09-14', '2020-10-11', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2020-10-12', '2020-11-08', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2020-11-09', '2020-12-06', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2020-12-07', '2021-01-03', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2021-01-04', '2021-01-31', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2021-02-01', '2021-02-28', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2021-03-01', '2021-03-28', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2021-03-29', '2021-04-25', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2021-04-26', '2021-05-23', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2021-05-24', '2021-06-20', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2021-06-21', '2021-07-18', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2021-07-19', '2021-08-15', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2021-08-16', '2021-09-12', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2021-09-13', '2021-10-10', 'mmsi_ship_register_2021-06-09.parquet'),
    ('2021-10-11', '2021-11-07', 'mmsi_ship_register_2022-02-28.parquet'),
    ('2021-11-08', '2021-12-05', 'mmsi_ship_register_2022-02-28.parquet'),
    ('2021-12-06', '2022-01-02', 'mmsi_ship_register_2022-02-28.parquet'),
    ('2022-01-03', '2022-01-30', 'mmsi_ship_register_2022-02-28.parquet'),
    ('2022-01-31', '2022-02-27', 'mmsi_ship_register_2022-02-28.parquet'),
    ('2022-02-28', '2022-03-27', 'mmsi_ship_register_2022-03-28.parquet'),
    ('2022-03-28', '2022-04-24', 'mmsi_ship_register_2022-05-02.parquet'),
    ('2022-04-25', '2022-05-22', 'mmsi_ship_register_2022-05-30.parquet'),
    ('2022-05-23', '2022-06-19', 'mmsi_ship_register_2022-06-27.parquet'),
    ('2022-06-20', '2022-07-17', 'mmsi_ship_register_2022-08-01.parquet'),
    ('2022-07-18', '2022-08-14', 'mmsi_ship_register_2022-08-01.parquet'),
    ('2022-08-15', '2022-09-11', 'mmsi_ship_register_2022-08-29.parquet'),
    ('2022-09-12', '2022-10-09', 'mmsi_ship_register_2022-09-26.parquet'),
    ('2022-10-10', '2022-11-06', 'mmsi_ship_register_2022-10-31.parquet'),
    ('2022-11-07', '2022-12-04', 'mmsi_ship_register_2022-11-28.parquet'),
    ('2022-12-05', '2023-01-01', 'mmsi_ship_register_2023-01-02.parquet'),
    ('2023-01-02', '2023-01-29', 'mmsi_ship_register_2023-01-30.parquet'),
    ('2023-01-30', '2023-02-26', 'mmsi_ship_register_2023-02-27.parquet'),
    ('2023-02-27', '2023-03-26', 'mmsi_ship_register_2023-03-27.parquet'),
    ('2023-03-27', '2023-04-23', 'mmsi_ship_register_2023-05-01.parquet'),
    ('2023-04-24', '2023-05-21', 'mmsi_ship_register_2023-05-29.parquet'),
    ('2023-05-22', '2023-06-18', 'mmsi_ship_register_2023-06-26.parquet'),
    ('2023-06-19', '2023-07-16', 'mmsi_ship_register_2023-07-31.parquet'),
    ('2023-07-17', '2023-08-13', 'mmsi_ship_register_2023-07-31.parquet'),
    ('2023-08-14', '2023-09-10', 'mmsi_ship_register_2023-08-28.parquet'),
    ('2023-09-11', '2023-10-08', 'mmsi_ship_register_2023-10-02.parquet'),
    ('2023-10-09', '2023-11-05', 'mmsi_ship_register_2023-10-30.parquet'),
    ('2023-11-06', '2023-12-03', 'mmsi_ship_register_2023-11-27.parquet'),
    ('2023-12-04', '2023-12-31', 'mmsi_ship_register_2024-01-01.parquet'),
    ('2024-01-01', '2024-01-28', 'mmsi_ship_register_2024-01-29.parquet'),
    ('2024-01-29', '2024-02-25', 'mmsi_ship_register_2024-02-26.parquet'),
    ('2024-02-26', '2024-03-24', 'mmsi_ship_register_2024-04-01.parquet'),
    ('2024-03-25', '2024-04-21', 'mmsi_ship_register_2024-04-29.parquet'),
    ('2024-04-22', '2024-05-19', 'mmsi_ship_register_2024-05-27.parquet'),
    ('2024-05-20', '2024-06-16', 'mmsi_ship_register_2024-07-01.parquet'),
    ('2024-06-17', '2024-07-14', 'mmsi_ship_register_2024-07-01.parquet'),
    ('2024-07-15', '2024-08-11', 'mmsi_ship_register_2024-07-29.parquet'),
    ('2024-08-12', '2024-09-08', 'mmsi_ship_register_2024-09-02.parquet'),
    ('2024-09-09', '2024-10-06', 'mmsi_ship_register_2024-09-30.parquet'),
    ('2024-10-07', '2024-11-03', 'mmsi_ship_register_2024-10-28.parquet'),
    ('2024-11-04', '2024-12-01', 'mmsi_ship_register_2024-12-02.parquet'),
    ('2024-12-02', '2024-12-29', 'mmsi_ship_register_2024-12-30.parquet'),
    ('2024-12-30', '2025-01-26', 'mmsi_ship_register_2025-01-27.parquet'),
    ('2025-01-27', '2025-02-23', 'mmsi_ship_register_2025-02-24.parquet'),
    ('2025-02-24', '2025-03-23', 'mmsi_ship_register_2025-03-31.parquet'),
    ('2025-03-24', '2025-04-20', 'mmsi_ship_register_2025-04-28.parquet'),
    ('2025-04-21', '2025-05-18', 'mmsi_ship_register_2025-06-02.parquet'),
    ('2025-05-19', '2025-06-15', 'mmsi_ship_register_2025-06-02.parquet'),
    ('2025-06-16', '2025-07-13', 'mmsi_ship_register_2025-06-30.parquet'),
    ('2025-07-14', '2025-08-10', 'mmsi_ship_register_2025-07-28.parquet'),
    ('2025-08-11', '2025-09-07', 'mmsi_ship_register_2025-09-01.parquet'),
    ('2025-09-08', '2025-10-05', 'mmsi_ship_register_2025-09-29.parquet'),
    ('2025-10-06', '2025-11-02', 'mmsi_ship_register_2025-10-27.parquet'),
    ('2025-11-03', '2025-11-30', 'mmsi_ship_register_2025-12-01.parquet'),
    ('2025-12-01', '2025-12-28', 'mmsi_ship_register_2025-12-29.parquet'),
    ('2025-12-29', '2026-01-25', 'mmsi_ship_register_2026-02-02.parquet'),
    ('2026-01-26', '2026-02-22', 'mmsi_ship_register_2026-03-02.parquet'),
    ('2026-02-23', '2026-03-22', 'mmsi_ship_register_2026-03-30.parquet'),
    ('2026-03-23', '2026-04-19', 'mmsi_ship_register_2026-04-27.parquet'),
    ('2026-04-20', '2026-05-17', 'mmsi_ship_register_2026-04-27.parquet'),
    ('2026-05-18', '2026-06-14', 'mmsi_ship_register_2026-04-27.parquet'),
]

## Configuration

In [5]:
user = 'onsfi'
country_code = 'GB'
statcode5 = 'A'
mangle = False

CENTRAL_SR_BASE = "s3a://datalab-142496269814-user-bucket/user-sc5k-ochag/ship_register"
OUTPUT_BASE = Path('./output')
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

report_plan = '''title,filename,vessel_types,arrival_key,location_key,aggfield,aggfunc
Daily all visits,daily_all_visits.csv,"['Cargo', 'Carrier', 'Tanker', 'Passenger']",date_arrival,reporting_locode,imo,count
Daily C&T visits,daily_cargotanker_visits.csv,"['Cargo', 'Carrier', 'Tanker']",date_arrival,reporting_locode,imo,count
Daily all unique ships,daily_all_unique_ships.csv,"['Cargo', 'Carrier', 'Tanker', 'Passenger']",date_arrival,reporting_locode,imo,nunique
Daily C&T unique ships,daily_cargotanker_unique_ships.csv,"['Cargo', 'Carrier', 'Tanker']",date_arrival,reporting_locode,imo,nunique
Weekly all visits,weekly_all_visits.csv,"['Cargo', 'Carrier', 'Tanker', 'Passenger']",week_arrival,reporting_locode,imo,count
Weekly C&T visits,weekly_cargotanker_visits.csv,"['Cargo', 'Carrier', 'Tanker']",week_arrival,reporting_locode,imo,count
Weekly all unique ships,weekly_all_unique_ships.csv,"['Cargo', 'Carrier', 'Tanker', 'Passenger']",week_arrival,reporting_locode,imo,nunique
Weekly C&T unique ships,weekly_cargotanker_unique_ships.csv,"['Cargo', 'Carrier', 'Tanker']",week_arrival,reporting_locode,imo,nunique
Passenger Daily Visits,daily_passenger_visits.csv,['Passenger'],date_arrival,reporting_locode,imo,count
Tanker Weekly Gross Tonnage,tanker_weekly_grosstonnage.csv,['Tanker'],week_arrival,reporting_locode,GrossTonnage,sum
Cargo Weekly TEUs,cargo_weekly_teu.csv,['Cargo'],week_arrival,reporting_locode,TEU,sum
Cargo Weekly Mean Time in Port,cargo_weekly_meantime.csv,['Cargo'],week_arrival,reporting_locode,time_in_port,mean
'''

report_plan_df = pd.read_csv(StringIO(report_plan))[
    ['title', 'filename', 'vessel_types', 'arrival_key', 'location_key', 'aggfield', 'aggfunc']
]

## Initialise Spark (once — reused across all 97 windows)

In [6]:
spark = (
    SparkSession
    .builder
    .appName('BatchShippingIndicators')
    .getOrCreate()
)

logger = utils.setup_logging()
warnings.simplefilter(action='ignore', category=FutureWarning)

## Load Fixed Assets

These do not change across batches: port zones, KNN model, cloudpickle UDFs,
and the FI reporting mappings.

In [7]:
port_zones_gdf = ports.load_from_package()
port_zones_sdf = spark.createDataFrame(port_zones_gdf.astype({"port_zone": str}))
ports_knn = port_zones_gdf.pipe(ports.explode_port_geometry).pipe(ports.fit_knn)
logger.info(f'Port zones loaded — global: {len(port_zones_gdf)}, GB: {(port_zones_gdf.locode.str[:2] == "GB").sum()}')

uk_ais_h3_filter = partial(
    positions.ais_h3_filter_by_locode,
    port_zones_gdf=port_zones_gdf,
    locodes=['GBFXT', 'GBLER', 'NLRTM']
)

cloudpickle.register_pickle_by_value(pc_common_pandas)
cloudpickle.register_pickle_by_value(pc_common_spark)
find_nearest_port = partial(
    pickle.loads(cloudpickle.dumps(pc_common_spark.find_nearest_port)),
    ports_knn=ports_knn
)
add_haversine_distance = pickle.loads(cloudpickle.dumps(pc_common_spark.add_haversine_distance))

fi_ship_types_map_df = reporting.load_ship_type_mapping()
fi_locode_map_df = reporting.load_fi_locode_mapping().rename(columns={'fi_locode': 'reporting_locode'})
fi_port_locodes = ports.load_from_package(file="ports_fi_boxes.csv").locode.to_list()

logger.info(f'FI ship types: {len(fi_ship_types_map_df)}, FI locodes: {len(fi_port_locodes)}')

/opt/python/lib/python3.11/site-packages/sklearn/neighbors/_base.py:501: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)
2026-06-09 10:15:42,413 - ungp_port_calls - INFO - Port zones loaded — global: 2979, GB: 330
2026-06-09 10:15:42,432 - ungp_port_calls - INFO - FI ship types: 70, FI locodes: 16


## Batch Loop

Progress is logged after each window. If a window fails, its partial output
directory is removed so it will be retried on the next run.

In [ ]:
failed_runs = []
total = len(BATCH_SCHEDULE)

for batch_idx, (start_date, end_date, assigned_ship_register) in enumerate(BATCH_SCHEDULE, 1):
    window_name = f'{start_date}_{end_date}'
    output_dir = OUTPUT_BASE / window_name
    sentinel = output_dir / 'fi_port_calls.csv'

    if sentinel.exists():
        logger.info(f'[{batch_idx}/{total}] SKIP  {window_name} (already complete)')
        continue

    logger.info(f'[{batch_idx}/{total}] START {window_name}  reg={assigned_ship_register}')
    t_wall_start = time.perf_counter()
    t_calc_start = pd.Timestamp.now()

    ship_register_sdf = None
    port_calls_sdf = None

    try:
        output_dir.mkdir(parents=True, exist_ok=True)

        # ── Date setup ───────────────────────────────────────────────────────
        # ais_start_date is 1 week before reporting start to avoid cold-start
        # false positives (matches ui.get_date_selections lead_weeks=1 default)
        reporting_start_date = start_date
        ais_start_date = (pd.Timestamp(start_date) - pd.Timedelta('7D')).date().isoformat()

        # ── Ship Register ────────────────────────────────────────────────────
        mmsi_sr_map_path = f"{CENTRAL_SR_BASE}/{assigned_ship_register}"
        ship_register_sdf = ships.load_mmsi_to_ship_register_mapping(spark, mmsi_sr_map_path)
        ships_sdf = (
            ships.select_ships_by_statcode5(ship_register_sdf, statcode5)
            .select('mmsi', 'MaritimeMobileServiceIdentityMMSINumber', 'LRIMOShipNo', 'StatCode5')
        )

        # ── AIS positions ────────────────────────────────────────────────────
        ais_df = (
            positions.load_ais_global_df(spark, ais_start_date, end_date)
            .transform(uk_ais_h3_filter)
            .drop('H3Index_0', 'message_type')
            .join(ships_sdf, on='mmsi', how='inner')
            .transform(positions.replace_ais_ship_identifiers)
            .transform(positions.fix_ais_data_types)
        )
        t_aisdf_done = pd.Timestamp.now()

        # ── Port call detection ──────────────────────────────────────────────
        port_calls_sdf = (
            ais_df
            .transform(identify_port_call_candidates)
            .transform(find_nearest_port)
            .join(port_zones_sdf, on="locode", how="left")
            .transform(add_haversine_distance)
            .transform(check_vessel_in_port_zone)
            .transform(dedup_port_calls)
            .transform(redact_startend_times)
            .dropna(subset='time_arrival')
            .join(broadcast(ship_register_sdf.drop('StatCode5', 'vessel_name')), on='mmsi', how='inner')
            .orderBy("imo", col("time_arrival").desc())
            .select(*column_sets['reporting'])
            .cache()
        )
        ct_port_calls = port_calls_sdf.count()
        t_portcalls_done = pd.Timestamp.now()
        logger.info(f'[{batch_idx}/{total}]  port visits: {ct_port_calls}')

        # ── Vessel filtering ─────────────────────────────────────────────────
        vessels_active_sdf = reporting.identify_active_vessels_for_country(port_calls_sdf, country_code)
        ct_vessels_active = vessels_active_sdf.count()

        port_calls_by_active_vessels_df = reporting.get_port_calls_by_active_vessels(
            port_calls_sdf, vessels_active_sdf, port_zones_sdf
        )
        ct_port_calls_by_active_vessels = port_calls_by_active_vessels_df.imo.count()

        vessel_activity_df = reporting.summarise_vessel_activity(port_calls_by_active_vessels_df)
        ct_vessels_active_gb = len(vessel_activity_df)

        vessels_excluded_df = reporting.select_excluded_vessels(vessel_activity_df)
        excluded_imos = vessels_excluded_df.imo.to_list()
        ct_excluded_imos = len(excluded_imos)
        vessels_excluded_df.to_csv(output_dir / 'vessels_excluded.csv', index=False)

        # ── FI port calls (pandas) ───────────────────────────────────────────
        fi_port_calls_df = (
            port_calls_sdf
            .filter(col('locode').startswith(country_code))
            .filter(col('in_port_zone') == True)
            .filter(~col('imo').isin(excluded_imos))
            .filter(col("time_arrival") > reporting_start_date)
            .limit(100000)
            .toPandas()
            .pipe(reporting.time_arrival_to_dates)
            .merge(fi_ship_types_map_df[['StatCode5', 'fi_vessel_type']], on='StatCode5', how='left')
            .pipe(reporting.add_reporting_ports, fi_locode_map_df)
            .query(f'reporting_locode in {fi_port_locodes}')
            .reset_index(drop=True)
        )
        ct_fi_port_calls = len(fi_port_calls_df)
        t_pandasout_done = pd.Timestamp.now()
        logger.info(f'[{batch_idx}/{total}]  FI port calls: {ct_fi_port_calls}')

        # Save raw FI port calls (also acts as completion sentinel)
        fi_port_calls_df.to_csv(output_dir / 'fi_port_calls.csv', index=False)

        # ── Aggregated reports ───────────────────────────────────────────────
        for _, row in report_plan_df.iterrows():
            _title, report_df = reporting.make_report(row, fi_port_calls_df, fi_mangle=mangle, output_link=False)
            report_df.reset_index().to_csv(output_dir / row['filename'], index=False)

        # ── Traffic detail ───────────────────────────────────────────────────
        visits_daily_df = reporting.visits_by_port_vessel(fi_port_calls_df)
        visits_daily_df.to_csv(output_dir / 'all_daily_traffic.csv', index=False)

        visits_weekly_df = reporting.visits_by_port_vessel(fi_port_calls_df, arrival_key='week_arrival')
        visits_weekly_df.to_csv(output_dir / 'all_weekly_traffic.csv', index=False)

        # ── Sanity / timing checks ───────────────────────────────────────────
        dic_checks = {
            'Port calls': ct_port_calls,
            'Active vessels': ct_vessels_active,
            'Port calls by active vessels': ct_port_calls_by_active_vessels,
            'Vessels active in UK': ct_vessels_active_gb,
            'Excluded IMOs': ct_excluded_imos,
            'FI Ship types map': len(fi_ship_types_map_df),
            'FI LOCODE map': len(fi_locode_map_df),
            'FI port LOCODEs': len(fi_port_locodes),
            'FI port calls': ct_fi_port_calls,
        }
        dic_times = {
            'Calculation start': t_calc_start,
            'AIS df complete': t_aisdf_done,
            'Port calls complete': t_portcalls_done,
            'Pandas DataFrame generated': t_pandasout_done,
        }
        livechecks.sanity_check(dic_checks).to_csv(output_dir / 'sanity_checks.csv')
        livechecks.time_check(dic_times).to_csv(output_dir / 'time_checks.csv', index=False)

        elapsed = time.perf_counter() - t_wall_start
        logger.info(f'[{batch_idx}/{total}] DONE  {window_name} — {elapsed:.0f}s')

    except Exception as exc:
        elapsed = time.perf_counter() - t_wall_start
        logger.error(f'[{batch_idx}/{total}] FAIL  {window_name} after {elapsed:.0f}s: {exc}')
        failed_runs.append({'window': window_name, 'error': str(exc)})
        # Remove partial output so the window is retried on the next run
        if output_dir.exists() and not sentinel.exists():
            import shutil
            shutil.rmtree(output_dir, ignore_errors=True)

    finally:
        # Always free Spark cache for per-batch DataFrames
        if port_calls_sdf is not None:
            port_calls_sdf.unpersist()
        if ship_register_sdf is not None:
            ship_register_sdf.unpersist()

2026-06-09 10:15:47,367 - ungp_port_calls - INFO - [1/97] START 2019-01-07_2019-02-03  reg=mmsi_ship_register_2021-06-09.parquet


## Summary

In [ ]:
total_done = sum(
    1 for start, end, _ in BATCH_SCHEDULE
    if (OUTPUT_BASE / f'{start}_{end}' / 'fi_port_calls.csv').exists()
)
logger.info(f'Batch complete — {total_done}/{total} windows done, {len(failed_runs)} failed')

if failed_runs:
    failed_df = pd.DataFrame(failed_runs)
    failed_df.to_csv(OUTPUT_BASE / 'failed_runs.csv', index=False)
    print(f'\n{len(failed_runs)} window(s) failed:')
    display(failed_df)
else:
    print(f'All {total_done} windows completed successfully.')